# core

> Decomposition step state management helpers

In [ ]:
#| default_exp routes.decomposition.core

In [ ]:
#| export
from typing import List, Dict, Any, NamedTuple

from cjm_fasthtml_card_stack.core.models import CardStackState
from cjm_fasthtml_card_stack.core.constants import DEFAULT_VISIBLE_COUNT, DEFAULT_CARD_WIDTH

from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_workflow_state.history import push_history

from cjm_fasthtml_workflow_transcript_decomp.core.models import (
    WorkingSegment, DecompositionStepState
)
from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

## State Management Helpers

In [ ]:
#| export
class DecompContext(NamedTuple):
    """Common decomposition state values loaded by handlers."""
    segment_dicts: List[Dict[str, Any]]  # Serialized working segments
    focused_index: int  # Currently focused segment index
    visible_count: int  # Number of visible cards in viewport
    card_width: int  # Card stack width in rem
    history: list  # Undo history stack

In [ ]:
#| export
def _to_segments(
    segment_dicts: List[Dict[str, Any]]  # Serialized segment dictionaries
) -> List[WorkingSegment]:  # Deserialized WorkingSegment objects
    """Convert segment dictionaries to WorkingSegment objects."""
    return [WorkingSegment.from_dict(s) for s in segment_dicts]

In [ ]:
#| export
def _get_decomp_state(
    workflow: StructureDecompWorkflow,  # The workflow instance
    session_id: str  # Session identifier string
) -> DecompositionStepState:  # Decomposition step state dictionary
    """Get the decomposition step state from the workflow state store."""
    workflow_state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
    step_states = workflow_state.get("step_states", {})
    return step_states.get("decomposition", {})

In [ ]:
#| export
def _get_selection_state(
    workflow: StructureDecompWorkflow,  # The workflow instance
    session_id: str  # Session identifier string
) -> Dict[str, Any]:  # Selection step state dictionary
    """Get the selection step state (Phase 1) from the workflow state store."""
    workflow_state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
    step_states = workflow_state.get("step_states", {})
    return step_states.get("selection", {})

In [ ]:
#| export
def _build_card_stack_state(
    ctx: DecompContext,  # Loaded decomposition context
    active_mode: str = None,  # Active interaction mode (e.g. "split")
) -> CardStackState:  # Card stack state for library functions
    """Build a CardStackState from decomposition context for library calls."""
    return CardStackState(
        focused_index=ctx.focused_index,
        visible_count=ctx.visible_count,
        card_width=ctx.card_width,
        active_mode=active_mode,
    )

In [ ]:
#| export
def _load_decomp_context(
    workflow: StructureDecompWorkflow,  # The workflow instance
    session_id: str  # Session identifier string
) -> DecompContext:  # Common decomposition state values
    """Load commonly-needed decomposition state values in a single call."""
    decomp_state = _get_decomp_state(workflow, session_id)
    return DecompContext(
        segment_dicts=decomp_state.get("segments", []),
        focused_index=decomp_state.get("focused_index", 0),
        visible_count=decomp_state.get("visible_count", DEFAULT_VISIBLE_COUNT),
        card_width=decomp_state.get("card_width", DEFAULT_CARD_WIDTH),
        history=decomp_state.get("history", []),
    )

In [ ]:
#| export
def _update_decomp_state(
    workflow: StructureDecompWorkflow,  # The workflow instance
    session_id: str,  # Session identifier string
    segments: List[Dict[str, Any]] = None,  # Updated segments (None = don't change)
    initial_segments: List[Dict[str, Any]] = None,  # Initial segments for reset (None = don't change)
    focused_index: int = None,  # Updated focused index (None = don't change)
    is_initialized: bool = None,  # Initialization flag (None = don't change)
    history: List[Dict[str, Any]] = None,  # Updated history (None = don't change)
    visible_count: int = None,  # Visible card count (None = don't change)
    card_width: int = None,  # Card stack width in rem (None = don't change)
) -> None:
    """Update the decomposition step state in the workflow state store."""
    workflow_state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
    step_states = workflow_state.get("step_states", {})
    decomp_state = step_states.get("decomposition", {})
    
    # Update only provided fields
    if segments is not None:
        decomp_state["segments"] = segments
    if initial_segments is not None:
        decomp_state["initial_segments"] = initial_segments
    if focused_index is not None:
        decomp_state["focused_index"] = focused_index
    if is_initialized is not None:
        decomp_state["is_initialized"] = is_initialized
    if history is not None:
        decomp_state["history"] = history
    if visible_count is not None:
        decomp_state["visible_count"] = visible_count
    if card_width is not None:
        decomp_state["card_width"] = card_width
    
    step_states["decomposition"] = decomp_state
    workflow_state["step_states"] = step_states
    workflow.state_store.update_state(workflow.config.workflow_id, session_id, workflow_state)

In [ ]:
#| export
def _push_history(
    workflow: StructureDecompWorkflow,  # The workflow instance
    session_id: str,  # Session identifier string
    current_segments: List[Dict[str, Any]],  # Current segments to snapshot
    focused_index: int,  # Current focused index to snapshot
) -> int:  # New history depth after push
    """Push current state to history stack before making changes."""
    decomp_state = _get_decomp_state(workflow, session_id)
    history = decomp_state.get("history", [])
    
    snapshot = {"segments": current_segments, "focused_index": focused_index}
    new_history = push_history(history, snapshot, workflow.config.max_history_depth)
    _update_decomp_state(workflow, session_id, history=new_history)
    return len(new_history)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()